# 04 — Model Selection & Model Training & Model Evaluation

**Input:** `features_listing.csv` (via `FEATURES_PATH` in config)  
**Target:** `sold_price`  
**Dataset shape:** 12,017 rows × 35 columns (34 features + target)

**Notebook covers:**
1. Load feature dataset and split into train/test
2. Evaluation function
3. Baseline models — OLS, Ridge, Lasso
4. Cross-validation stability check
5. Feature selection — Sequential Forward Selection (SFS) and Backward Elimination (BES)
6. Final model on selected features + coefficients

## Setup

Import libraries, load config, and read the final feature dataset produced by the preprocessing notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from  sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, KFold

%run config.ipynb

df = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

Shape: (12017, 35)
Columns: ['week_of_month', 'listing_price', 'bathrooms', 'parking_spaces', 'encoded_house_age', 'property_type_Common Element Condo', 'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_House', 'property_type_Link', 'property_type_Others', 'property_type_SEMI-DETACHED', 'city_Aurora', 'city_Brampton', 'city_Brock', 'city_Burlington', 'city_Caledon', 'city_Clarington', 'city_Durham', 'city_Halton', 'city_Halton Hills', 'city_King', 'city_Markham', 'city_Mississauga', 'city_Oshawa', 'city_Peel', 'city_Pickering', 'city_Richmond Hill', 'city_Scugog', 'city_Vaughan', 'city_Whitby', 'city_Whitchurch-stouffville', 'city_York', 'sold_price']


## 1. Train / Test Split

Separating features (`X`) from the target (`Y`), scaling with `StandardScaler`, then splitting 80/20 into train and test sets with `random_state=42` for reproducibility.

**Important:** The scaler is fitted on the full dataset here for baseline evaluation. The final model uses a separate scaler fitted only on the selected features — see Section 6.

In [2]:
X = df.drop(columns = ['sold_price'])
Y = df['sold_price']

scaler = StandardScaler()
x_scaled = scaler.fit_transform(X)

x_train, x_test, y_train, y_test = train_test_split(x_scaled,Y, test_size = 0.2, random_state=42)

print(f'Train: {x_train.shape}   | Test: {x_test.shape}')

Train: (9613, 34)   | Test: (2404, 34)


## 2. Evaluation Function

Reusable function that fits a model and prints four metrics:

- **MAPE** — Mean Absolute Percentage Error (lower is better, target < 5%)
- **R²** — proportion of variance explained (higher is better, target > 0.98)
- **Within 10%** — % of predictions within 10% of actual sold price
- **Within 15%** — % of predictions within 15% of actual sold price

In [3]:
def evalute(name, model, tr_x = x_train, te_x = x_test, tr_y = y_train, te_y = y_test):
    model.fit(tr_x, tr_y)
    pred = model.predict(te_x)

    mape = mean_absolute_percentage_error(y_test, pred) * 100
    r2 = r2_score(y_test, pred)

    devination_10 = (np.abs(pred - y_test) / y_test <= 0.10).mean() * 100
    devination_15 = (np.abs(pred - y_test) / y_test <= 0.15).mean() * 100

    print(f"--{name} Model---")
    print(f"Mape:  {mape:.2f}%")
    print(f"R^2     {r2:.4f}")
    print(f"Within 10:  {devination_10:.1f}%")
    print(f"Within 15:  {devination_15:.1f}%")
    return model

## 3. Baseline Models — OLS, Ridge, Lasso

Running three linear models on the full feature set as a baseline before feature selection:

- **OLS** (Ordinary Least Squares) — standard linear regression, no regularization
- **Ridge** — L2 regularization, penalizes large coefficients, tested across alpha values `[0.01, 0.1, 1, 10, 100, 1000]`
- **Lasso** — L1 regularization, can shrink coefficients to zero (implicit feature selection)

All three are expected to perform similarly given the high correlation between `listing_price` and `sold_price`.

In [4]:
ols_model = evalute('OLS', LinearRegression())
ridge_model = evalute("Ridge", RidgeCV(alphas = [0.01, 0.1, 1, 10, 100, 1000]))
lasso_model = evalute('Lasso', LassoCV(cv =5, random_state =42, max_iter = 1000))

--OLS Model---
Mape:  3.85%
R^2     0.9867
Within 10:  92.7%
Within 15:  98.0%
--Ridge Model---
Mape:  3.85%
R^2     0.9867
Within 10:  92.7%
Within 15:  98.0%
--Lasso Model---
Mape:  3.84%
R^2     0.9867
Within 10:  92.6%
Within 15:  97.9%


## 4. Cross-Validation Stability Check

5-fold cross-validation on the full feature set using OLS. This checks whether the R² score is stable across different data splits — a low standard deviation means the model generalises well and is not overfitting to a particular train/test split.

**Target:** std < 0.005

In [5]:
cv = KFold(n_splits = 5 ,shuffle = True, random_state=42)
cv_scores = cross_val_score(LinearRegression(), x_scaled, Y, cv=cv, scoring ='r2')
print(f"CV R^2 scores: {cv_scores.round(4)}")
print(f"Mean: {cv_scores.mean():.4f}   | STD: {cv_scores.std():.4f}")

CV R^2 scores: [0.9867 0.9861 0.9856 0.9855 0.9861]
Mean: 0.9860   | STD: 0.0004


## 5. Feature Selection — SFS and BES

Two automated feature selection methods are run to identify the most predictive subset of features:

- **SFS (Sequential Forward Selection)** — starts with no features and adds the best one at each step
- **BES (Backward Elimination Selection)** — starts with all features and removes the least useful one at each step

Both use 5-fold CV with R² as the scoring metric. Features selected by both methods represent the most robust subset. The union of both sets is used for the final model.

In [6]:
# SFS

sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select='auto',
    direction='forward',
    scoring='r2',
    cv=5
)

# BES

bes = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select='auto',
    direction='backward',
    scoring='r2',
    cv=5
)

sfs.fit(x_scaled, Y)
bes.fit(x_scaled, Y)

fwd_features = X.columns[sfs.get_support()].tolist()
bes_features = X.columns[bes.get_support()].tolist()
print(f'SFS selected {len(fwd_features)}: {fwd_features}')
print(f'SFS selected {len(bes_features)}: {bes_features}')
print(f'In both: {set(fwd_features) & set(fwd_features)}')

SFS selected 17: ['listing_price', 'bathrooms', 'property_type_Common Element Condo', 'property_type_Condo Apartment', 'property_type_Condo Townhouse', 'property_type_Detached', 'property_type_House', 'property_type_Link', 'city_Aurora', 'city_Brampton', 'city_Brock', 'city_Caledon', 'city_Halton Hills', 'city_King', 'city_Pickering', 'city_Whitby', 'city_York']
SFS selected 17: ['listing_price', 'bathrooms', 'property_type_Condo Apartment', 'property_type_Detached', 'property_type_Link', 'property_type_SEMI-DETACHED', 'city_Brampton', 'city_Burlington', 'city_Caledon', 'city_Durham', 'city_Halton', 'city_Markham', 'city_Mississauga', 'city_Oshawa', 'city_Peel', 'city_Scugog', 'city_York']
In both: {'property_type_Link', 'property_type_Condo Apartment', 'city_Aurora', 'city_Whitby', 'city_Brock', 'listing_price', 'city_King', 'property_type_Common Element Condo', 'city_Brampton', 'city_Halton Hills', 'city_Pickering', 'property_type_House', 'property_type_Condo Townhouse', 'city_York',

## 6. Final Model — Selected Features + Coefficients

Building the final OLS model on the union of SFS and BES selected features. A fresh `StandardScaler` is fitted only on the selected feature columns — this is the scaler that gets saved and used in the Streamlit deployment app.

Coefficients are printed for interpretability — the magnitude of each coefficient (in the scaled space) indicates the relative importance of each feature to the predicted sold price.

In [7]:
final_set  = list(set(fwd_features) | set(bes_features))
final_model_x = df[final_set]
final_scaler = StandardScaler()
final_x_scaled = final_scaler.fit_transform(final_model_x)

tr_x, te_x, tr_y, te_y =  train_test_split(final_x_scaled,Y, test_size = 0.2, random_state=42)

model_final = evalute("Final OLS", LinearRegression(),tr_x, te_x, tr_y, te_y)

print("Coefficients:")
for word, coef in zip(final_set, model_final.coef_):
    print(f"  {word}: {coef:,.2f}")
print(f" Intercept: {model_final.intercept_:,.2f}")

--Final OLS Model---
Mape:  3.84%
R^2     0.9867
Within 10:  92.7%
Within 15:  97.8%
Coefficients:
  city_Whitby: -0.00
  city_Halton: 3,399.22
  city_Pickering: 438.68
  property_type_SEMI-DETACHED: 1,727.27
  property_type_Condo Townhouse: -2,667.66
  city_Scugog: 362.22
  property_type_Detached: 7,913.03
  city_Caledon: -886.12
  bathrooms: 18,256.96
  property_type_Link: 1,019.85
  property_type_Condo Apartment: -7,961.97
  city_Markham: -278.29
  city_Aurora: -0.00
  city_Burlington: 307.81
  city_Oshawa: -509.25
  city_Peel: -1,450.07
  listing_price: 523,641.25
  city_King: -579.68
  property_type_Common Element Condo: -1,374.10
  city_Brampton: -2,234.68
  city_Halton Hills: 821.83
  city_York: 12,207.45
  property_type_House: -0.00
  city_Durham: 1,803.81
  city_Mississauga: -443.60
  city_Brock: -1,024.22
 Intercept: 1,010,981.31
